In [7]:
import pandas as pd
import numpy as np
import os
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from datetime import timedelta

# --- Base Class ---
class BaseAnalyzer(object):
    def __init__(self, file_path, output_path, target_sensors):
        self.file_path = file_path
        self.output_path = output_path
        self.target_sensors = target_sensors
        self.df = None
        self.df_model = None

    def load_data(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"❌ فایل یافت نشد: {self.file_path}")
        self.df = pd.read_excel(self.file_path)
        if 'date' in self.df.columns:
            self.df['date'] = pd.to_datetime(self.df['date'])
        self.df_model = self.df.dropna(subset=self.target_sensors).copy()

    def preprocess(self):
        cols = []
        for sensor in self.target_sensors:
            name = f'{sensor}_smooth'
            self.df_model[name] = self.df_model[sensor].rolling(window=5, center=True).mean()
            cols.append(name)
        self.df_model = self.df_model.dropna(subset=cols).copy()
        return cols

    def save_output(self, final_df):
        try:
            os.makedirs(os.path.dirname(self.output_path), exist_ok=True)
            final_df.to_excel(self.output_path, index=False)
            print(f"✅ خروجی در مسیر زیر ذخیره شد:\n{self.output_path}")
        except Exception as e:
            print(f"❌ خطا در ذخیره: {e}")

# --- Derived Class with DBSCAN ---
class SmartBearingAnalyzer(BaseAnalyzer):
    def __init__(self, file_path, output_path, target_sensors, eps=0.5, min_samples=5):
        # مقداردهی مستقیم برای پایداری در نوت‌بوک
        self.file_path = file_path
        self.output_path = output_path
        self.target_sensors = target_sensors
        self.eps = eps
        self.min_samples = min_samples
        self.df = None
        self.df_model = None

    def run_analysis(self):
        self.load_data()
        smooth_cols = self.preprocess()

        # ۱. استانداردسازی
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(self.df_model[smooth_cols])

        # ۲. کلاسترینگ با DBSCAN
        model = DBSCAN(eps=self.eps, min_samples=self.min_samples)
        cluster_labels = model.fit_predict(scaled_data)
        self.df_model['Behavior_Cluster'] = cluster_labels

        # ۳. محاسبه معیار تخریب (فاصله تا نزدیک‌ترین نقطه هسته در همان خوشه)
        self.df_model['Degradation_Index'] = self._calculate_degradation_index(scaled_data, cluster_labels, model)

        # ۴. لیبل‌گذاری
        self.df_model['Health_Status'] = self.df_model.apply(self._labeling, axis=1)

        # ۵. فیلتر ۳۰ روز آخر
        if 'date' in self.df_model.columns:
            last_dt = self.df_model['date'].max()
            final_df = self.df_model[self.df_model['date'] >= (last_dt - timedelta(days=30))].copy()
        else:
            final_df = self.df_model

        self.save_output(final_df)

    def _calculate_degradation_index(self, scaled_data, cluster_labels, model):
        """
        محاسبه معیار تخریب:
        - برای نقاط هسته (core points): فاصله تا مرکز جرم خوشه خودشان
        - برای نقاط مرزی (border points): فاصله تا نزدیک‌ترین نقطه هسته در همان خوشه
        - برای نویز (noise, label=-1): حداکثر فاصله مشاهده شده + ۱ (به عنوان بدترین وضعیت)
        """
        n_samples = len(scaled_data)
        degradation = np.zeros(n_samples)
        
        # پیدا کردن نقاط هسته (core points) برای هر خوشه
        # در DBSCAN، نقاط هسته نقاطی هستند که حداقل min_samples نقطه در همسایگی eps دارند
        # ما از core_sample_indices_ استفاده می‌کنیم
        
        core_mask = np.zeros(n_samples, dtype=bool)
        core_mask[model.core_sample_indices_] = True
        
        # محاسبه مرکز جرم برای هر خوشه (فقط از نقاط هسته)
        unique_clusters = set(cluster_labels) - {-1}  # حذف نویز
        cluster_centers = {}
        
        for cluster_id in unique_clusters:
            cluster_core_points = scaled_data[(cluster_labels == cluster_id) & core_mask]
            if len(cluster_core_points) > 0:
                cluster_centers[cluster_id] = np.mean(cluster_core_points, axis=0)
            else:
                # اگر هیچ نقطه هسته‌ای در خوشه نبود، از میانگین کل نقاط خوشه استفاده کن
                cluster_all_points = scaled_data[cluster_labels == cluster_id]
                if len(cluster_all_points) > 0:
                    cluster_centers[cluster_id] = np.mean(cluster_all_points, axis=0)
                else:
                    cluster_centers[cluster_id] = np.zeros(scaled_data.shape[1])
        
        # محاسبه فاصله برای هر نقطه
        max_distance = 0
        
        for i in range(n_samples):
            cluster_id = cluster_labels[i]
            
            if cluster_id == -1:  # نویز
                # فعلاً صفر بگذار، بعداً مقداردهی می‌کنیم
                degradation[i] = 0
            else:
                if i in model.core_sample_indices_:
                    # نقطه هسته: فاصله تا مرکز جرم خوشه
                    center = cluster_centers.get(cluster_id, np.zeros(scaled_data.shape[1]))
                    degradation[i] = np.linalg.norm(scaled_data[i] - center)
                else:
                    # نقطه مرزی: فاصله تا نزدیک‌ترین نقطه هسته در همان خوشه
                    cluster_core_indices = [idx for idx in model.core_sample_indices_ 
                                           if cluster_labels[idx] == cluster_id]
                    
                    if len(cluster_core_indices) > 0:
                        core_points = scaled_data[cluster_core_indices]
                        distances_to_cores = np.linalg.norm(core_points - scaled_data[i], axis=1)
                        degradation[i] = np.min(distances_to_cores)
                    else:
                        # اگر هیچ نقطه هسته‌ای نبود، از مرکز جرم استفاده کن
                        center = cluster_centers.get(cluster_id, np.zeros(scaled_data.shape[1]))
                        degradation[i] = np.linalg.norm(scaled_data[i] - center)
            
            if degradation[i] > max_distance:
                max_distance = degradation[i]
        
        # مقداردهی به نقاط نویز (بدترین وضعیت)
        for i in range(n_samples):
            if cluster_labels[i] == -1:
                degradation[i] = max_distance + 1.0
        
        return degradation

    def _labeling(self, row):
        if row['Behavior_Cluster'] == -1:  # نقاط نویز
            return "Investigation Needed (Operational Drift)"
        elif row['Degradation_Index'] > 0.95:
            return "Investigation Needed (Operational Drift)"
        elif row['Degradation_Index'] > 0.85:
            return "Observation Required (Pattern Change)"
        else:
            return "Healthy (Optimal Performance)"

# --- CONFIG & RUN ---
CONFIG = {
    "file_path": r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx',
    "output_path": r'outputs\G11\dsas_g11_bearings_vibration_temp_clustering\clustering\dsas_g11_clustering_output2.xlsx',
    "sensors": ['AssetID_9368','AssetID_9369','AssetID_9370','AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
}

# اجرا با DBSCAN (پارامترهای eps و min_samples قابل تنظیم هستند)
analyzer = SmartBearingAnalyzer(
    CONFIG["file_path"], 
    CONFIG["output_path"], 
    CONFIG["sensors"],
    eps=0.5,      # شعاع همسایگی (قابل تنظیم بر اساس داده)
    min_samples=5  # حداقل تعداد نقاط برای تشکیل خوشه
)
analyzer.run_analysis()

✅ خروجی در مسیر زیر ذخیره شد:
outputs\G11\dsas_g11_bearings_vibration_temp_clustering\clustering\dsas_g11_clustering_output2.xlsx
